# 🔓 Uncensored AI in VM
Pipeline instan untuk menjalankan model AI Uncensored (**Gemma 31B**, **Qwen 14B**, atau **Qwen 7B**) di Cloud GPU Kaggle (2x T4) secara gratis dan menghubungkannya ke OpenCode / client lokal.

> **💡 Fleksibel & Otomatis:** Notebook ini mendukung model apa pun. Cukup pasang salah satu dataset model di `/kaggle/input`, kodingan di bawah otomatis mendeteksi dan menjalankannya!

---
### 📦 Sel 1: Instalasi Ollama Engine
Mengunduh dan memasang Ollama binary di container VM Kaggle.

In [ ]:
!apt-get update -qq > /dev/null 2>&1
!apt-get install -y -qq zstd 2>&1 | tail -1
!curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -6
!echo "---"
!ollama --version

---
### ⚡ Sel 2: Mount Model via Symlink (0 MB Disk) & Jalankan Server Ollama
- **Trik Symlink:** File bobot model (`blobs`) disambungkan via symlink langsung dari `/kaggle/input` sehingga memakan **0 MB kuota disk** (mencegah crash `Exit code: 137 / SIGKILL`).
- **Auto-Detect:** Otomatis mendeteksi model apa pun yang Anda pasang (Gemma 31B, Qwen 14B, atau Qwen 7B).
- **Konteks Besar:** `OLLAMA_CONTEXT_LENGTH=32768` (32k token) agar muat riwayat percakapan dan file coding panjang.
- **Keep-Alive:** Parameter `OLLAMA_KEEP_ALIVE=-1` menjaga model tetap standby di VRAM agar respon instan.

In [ ]:
# SEL 2: OLLAMA + MODEL RUNNER (0 MB DISK, ANTI-CRASH)
import subprocess, time, os, shutil

os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
os.environ["OLLAMA_ORIGINS"] = "*"
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"
os.environ["OLLAMA_CONTEXT_LENGTH"] = "32768"
os.environ["OLLAMA_MODELS"] = "/kaggle/temp/ollama-models"

# Bersihkan model lama
shutil.rmtree("/kaggle/temp/ollama-models", ignore_errors=True)
os.makedirs("/kaggle/temp/ollama-models/blobs", exist_ok=True)
os.makedirs("/kaggle/temp/ollama-models/manifests", exist_ok=True)

# 1. Cari dataset model di /kaggle/input
blobs_src = manifests_src = None
for root, dirs, files in os.walk("/kaggle/input"):
    for d in dirs:
        p = os.path.join(root, d)
        try:
            if d == "blobs" and os.listdir(p): blobs_src = p
            if d == "manifests" and os.listdir(p): manifests_src = p
        except PermissionError: pass

if blobs_src and manifests_src:
    # 2. Blobs (18.7 GB) -> SYMLINK (0 MB disk tambahan, anti-crash)
    print("Menghubungkan Blobs bobot model via symlink...")
    for f in os.listdir(blobs_src):
        sf = os.path.join(blobs_src, f)
        df = os.path.join("/kaggle/temp/ollama-models/blobs", f)
        if not os.path.exists(df):
            os.symlink(sf, df)
            
    # 3. Manifests (HANYA beberapa KB) -> COPY ASLI (Wajib agar Ollama tidak komplain)
    print("Menyalin Manifests (hanya beberapa KB)...")
    for root, dirs, files in os.walk(manifests_src):
        rel = os.path.relpath(root, manifests_src)
        tgt = os.path.join("/kaggle/temp/ollama-models/manifests", rel)
        os.makedirs(tgt, exist_ok=True)
        for f in files:
            shutil.copy2(os.path.join(root, f), os.path.join(tgt, f))
    print("Model siap ✅")
else:
    print("!! Dataset tidak ditemukan")

# 4. Restart Ollama server
subprocess.run(["pkill", "-f", "ollama serve"])
time.sleep(2)
print("Menyalakan server Ollama...")
subprocess.Popen(["ollama", "serve"], stdout=open("/tmp/ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(6)

r = subprocess.run(["ollama", "list"], capture_output=True, text=True)
print(r.stdout)
print("=== SERVER OLLAMA READY ===")

---
### 🌐 Sel 3: Tunnel Publik via Ngrok (Bebas Timeout 100s)
Membuka jalur tunnel publik ke Ollama port 11434. Tidak memiliki limit timeout 100 detik seperti Cloudflare.

> **Catatan:** Masukkan Authtoken Ngrok gratis Anda di variabel `NGROK_TOKEN` (ambil di [dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken)).

In [ ]:
# ==========================================
# SEL 3: TUNNEL NGROK (BEBAS TIMEOUT 100s)
# ==========================================
!pip install -q pyngrok
import time
from pyngrok import ngrok

NGROK_TOKEN = "YOUR_NGROK_AUTHTOKEN"

# 1. Set token
try:
    ngrok.set_auth_token(NGROK_TOKEN)
except Exception:
    pass

# 2. Tutup tunnel lama jika ada
ngrok.kill()

# 3. Hubungkan langsung ke port Ollama 11434
tunnel = ngrok.connect(11434)

print("\n" + "="*60)
print(f"🔥 NGROK URL ASLI:     {tunnel.public_url}")
print(f"🔥 URL UNTUK OPENCODE: {tunnel.public_url}/v1")

print("="*60 + "\n")
print("Tunnel aktif! Biarkan sel ini tetap berjalan [*]")

---
### 💬 Sel 4 (Opsional): Chat Interaktif Streaming Langsung di Notebook
Gunakan sel ini jika ingin langsung ngobrol dengan model di browser Kaggle tanpa perlu membuka client eksternal.
- Otomatis menggunakan model yang sedang aktif di Ollama (Gemma 31B atau Qwen 7B).
- Ketik `/reset` untuk hapus konteks riwayat obrolan.
- Ketik `exit` untuk mengakhiri chat.

In [ ]:
# ==========================================
# SEL CHAT INTERAKTIF DI NOTEBOOK KAGGLE
# ==========================================
import urllib.request, json, sys

# 1. Deteksi model yang sedang aktif di Ollama
try:
    req = urllib.request.Request("http://127.0.0.1:11434/api/tags")
    with urllib.request.urlopen(req) as resp:
        models = json.loads(resp.read().decode())["models"]
        model_name = models[0]["name"]
        print(f"🔥 Model aktif: {model_name}")
except Exception:
    model_name = "hf.co/mradermacher/gemma-4-31B-it-heretic-GGUF:Q4_K_M"
    print(f"Default model: {model_name}")

history = [
    {"role": "system", "content": "Kamu adalah asisten AI berbahasa Indonesia santai, cerdas, dan to the point."}
]

print("\n" + "="*50)
print("💬 CHAT DENGAN MODEL AKTIF (Ketik pesan lalu tekan Enter)")
print("Ketik '/reset' untuk hapus riwayat | Ketik 'exit' untuk stop")
print("="*50 + "\n")

while True:
    try:
        user_input = input("\nLu: ").strip()
    except (KeyboardInterrupt, EOFError):
        print("\n(Chat selesai)")
        break
    except Exception as e:
        print(f"Input error: {e}")
        break

    if not user_input:
        continue
    if user_input.lower() in ["exit", "quit", "keluar"]:
        print("(Chat selesai)")
        break
    if user_input.lower() == "/reset":
        history = history[:1]
        print("🧹 Konteks percakapan di-reset!\n")
        continue

    history.append({"role": "user", "content": user_input})
    print("\nAI: ", end="", flush=True)

    # Kirim ke Ollama dengan streaming real-time
    payload = json.dumps({
        "model": model_name,
        "messages": history,
        "stream": True
    }).encode("utf-8")
    
    req = urllib.request.Request(
        "http://127.0.0.1:11434/api/chat",
        data=payload,
        headers={"Content-Type": "application/json"}
    )

    ai_response = ""
    try:
        with urllib.request.urlopen(req) as resp:
            for line in resp:
                if line:
                    chunk = json.loads(line.decode("utf-8"))
                    content = chunk.get("message", {}).get("content", "")
                    ai_response += content
                    sys.stdout.write(content)
                    sys.stdout.flush()
        print("\n" + "-"*50)
        history.append({"role": "assistant", "content": ai_response})
    except Exception as e:
        print(f"\n[Gagal menghubungi Ollama: {e}]\n")

---
### 🛡️ Sel 5 (Opsional): Keep-Alive Penahan Tunnel (Khusus OpenCode Saja)
Jalankan sel ini **HANYA JIKA** Anda murni ingin memakai model dari **OpenCode** dan mendapati tunnel Ngrok sering tertutup sendiri saat notebook idle.

> ⚠️ **Perhatian:** Sel ini akan terus sibuk `[*]`. Jangan jalankan sel ini jika Anda ingin menggunakan **Sel 4 (Chat di Notebook)**.

In [ ]:
# SEL 5: KEEP-ALIVE LOOP (OPSIONAL - KHUSUS OPENCODE)
import time

print('🛡️ Keep-Alive aktif! Menjaga tunnel Ngrok tetap online untuk OpenCode.')
print('Tekan tombol Stop (kotak hitam) jika ingin menghentikan atau beralih ke Sel 4.')

while True:
    time.sleep(60)